Q1. Spark Initialization and Data Loading

In [1]:
import pyspark
print(f"PySpark version: {pyspark.__version__}")

PySpark version: 4.0.3


In [2]:
import os
os.environ['PYSPARK_PYTHON'] = 'python'
os.environ['PYSPARK_DRIVER_PYTHON'] = 'python'

from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder.appName('Bank Fraud')\
    .config('spark.python.worker.faulthandler.enabled', 'true')\
    .config('spark.python.worker.reuse', 'false')\
    .getOrCreate()

In [4]:
df = spark.read.format("csv").option("header","True").option("InferSchema","True").load("/content/Bank_Transaction_Fraud_Detection.csv")

In [5]:
df.printSchema()

root
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- State: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Bank_Branch: string (nullable = true)
 |-- Account_Type: string (nullable = true)
 |-- Transaction_ID: string (nullable = true)
 |-- Transaction_Date: string (nullable = true)
 |-- Transaction_Time: timestamp (nullable = true)
 |-- Transaction_Amount: double (nullable = true)
 |-- Merchant_ID: string (nullable = true)
 |-- Transaction_Type: string (nullable = true)
 |-- Merchant_Category: string (nullable = true)
 |-- Account_Balance: double (nullable = true)
 |-- Transaction_Device: string (nullable = true)
 |-- Transaction_Location: string (nullable = true)
 |-- Device_Type: string (nullable = true)
 |-- Is_Fraud: integer (nullable = true)
 |-- Transaction_Currency: string (nullable = true)
 |-- Customer_Contact: string (nullable = true)
 |-- 

In [6]:
df.show(5,truncate=False)

+------------------------------------+-------------------+------+---+-----------+------------------+-------------------------+------------+------------------------------------+----------------+-------------------+------------------+------------------------------------+----------------+-----------------+---------------+------------------+--------------------------+-----------+--------+--------------------+----------------+-----------------------+-----------------------+
|Customer_ID                         |Customer_Name      |Gender|Age|State      |City              |Bank_Branch              |Account_Type|Transaction_ID                      |Transaction_Date|Transaction_Time   |Transaction_Amount|Merchant_ID                         |Transaction_Type|Merchant_Category|Account_Balance|Transaction_Device|Transaction_Location      |Device_Type|Is_Fraud|Transaction_Currency|Customer_Contact|Transaction_Description|Customer_Email         |
+------------------------------------+--------------

Q2. RDD Implementation

In [7]:
import pyspark
import sys

print("PySpark:", pyspark.__version__)
print("Spark:", spark.version)
print("Python:", sys.version)

PySpark: 4.0.3
Spark: 4.0.3
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [8]:
import sys
import os

print(sys.executable)
print(os.path.exists(sys.executable))

/usr/bin/python3
True


In [9]:
# Convert DataFrame into RDD
transaction_rdd = df.rdd

print("RDD Created Successfully")
print(transaction_rdd)

RDD Created Successfully
MapPartitionsRDD[18] at javaToPython at NativeMethodAccessorImpl.java:0


In [10]:
total_transactions = transaction_rdd.count()

print("Total Transactions :", total_transactions)

Total Transactions : 61398


In [11]:
transaction_rdd.take(5)

[Row(Customer_ID='d5f6ec07-d69e-4f47-b9b4-7c58ff17c19e', Customer_Name='Osha Tella', Gender='Male', Age=60, State='Kerala', City='Thiruvananthapuram', Bank_Branch='Thiruvananthapuram Branch', Account_Type='Savings', Transaction_ID='4fa3208f-9e23-42dc-b330-844829d0c12c', Transaction_Date='23-01-2025', Transaction_Time=datetime.datetime(2026, 7, 3, 16, 4, 7), Transaction_Amount=32415.45, Merchant_ID='214e03c5-5c34-40d1-a66c-f440aa2bbd02', Transaction_Type='Transfer', Merchant_Category='Restaurant', Account_Balance=74557.27, Transaction_Device='Voice Assistant', Transaction_Location='Thiruvananthapuram, Kerala', Device_Type='POS', Is_Fraud=0, Transaction_Currency='INR', Customer_Contact='+9198579XXXXXX', Transaction_Description='Bitcoin transaction', Customer_Email='oshaXXXXX@XXXXX.com'),
 Row(Customer_ID='7c14ad51-781a-4db9-b7bd-67439c175262', Customer_Name='Hredhaan Khosla', Gender='Female', Age=51, State='Maharashtra', City='Nashik', Bank_Branch='Nashik Branch', Account_Type='Business'

In [12]:
for row in transaction_rdd.take(5):
    print(row)

Row(Customer_ID='d5f6ec07-d69e-4f47-b9b4-7c58ff17c19e', Customer_Name='Osha Tella', Gender='Male', Age=60, State='Kerala', City='Thiruvananthapuram', Bank_Branch='Thiruvananthapuram Branch', Account_Type='Savings', Transaction_ID='4fa3208f-9e23-42dc-b330-844829d0c12c', Transaction_Date='23-01-2025', Transaction_Time=datetime.datetime(2026, 7, 3, 16, 4, 7), Transaction_Amount=32415.45, Merchant_ID='214e03c5-5c34-40d1-a66c-f440aa2bbd02', Transaction_Type='Transfer', Merchant_Category='Restaurant', Account_Balance=74557.27, Transaction_Device='Voice Assistant', Transaction_Location='Thiruvananthapuram, Kerala', Device_Type='POS', Is_Fraud=0, Transaction_Currency='INR', Customer_Contact='+9198579XXXXXX', Transaction_Description='Bitcoin transaction', Customer_Email='oshaXXXXX@XXXXX.com')
Row(Customer_ID='7c14ad51-781a-4db9-b7bd-67439c175262', Customer_Name='Hredhaan Khosla', Gender='Female', Age=51, State='Maharashtra', City='Nashik', Bank_Branch='Nashik Branch', Account_Type='Business', T

In [13]:
for index, column in enumerate(df.columns):
    print(index, ":", column)

0 : Customer_ID
1 : Customer_Name
2 : Gender
3 : Age
4 : State
5 : City
6 : Bank_Branch
7 : Account_Type
8 : Transaction_ID
9 : Transaction_Date
10 : Transaction_Time
11 : Transaction_Amount
12 : Merchant_ID
13 : Transaction_Type
14 : Merchant_Category
15 : Account_Balance
16 : Transaction_Device
17 : Transaction_Location
18 : Device_Type
19 : Is_Fraud
20 : Transaction_Currency
21 : Customer_Contact
22 : Transaction_Description
23 : Customer_Email


In [14]:
mapped_rdd = transaction_rdd.map(
    lambda x: (x["Customer_ID"], x["Transaction_Amount"])
)

mapped_rdd.take(10)

[('d5f6ec07-d69e-4f47-b9b4-7c58ff17c19e', 32415.45),
 ('7c14ad51-781a-4db9-b7bd-67439c175262', 43622.6),
 ('3a73a0e5-d4da-45aa-85f3-528413900a35', 63062.56),
 ('7902f4ef-9050-4a79-857d-9c2ea3181940', 14000.72),
 ('3a4bba70-d9a9-4c5f-8b92-1735fd8c19e9', 18335.16),
 ('6c870d65-76b0-431d-bdf3-9292998e8211', 9711.15),
 ('5323737c-bbd2-423f-9c9b-e0433c8f75dc', 94677.01),
 ('c0c3d474-f6c2-4c66-9b0e-f9ba75c6f310', 67704.28),
 ('e9a82764-1253-4a46-ad34-80e3416fc801', 72953.45),
 ('708224d5-192a-4d86-b411-8ec6d1bb274b', 5689.02)]

In [15]:
fraud_rdd = transaction_rdd.filter(
    lambda x: x["Is_Fraud"] == 1
)

print("Fraud Transactions :", fraud_rdd.count())

Fraud Transactions : 3079


Q3. Key-Value Operations and Persistence

In [16]:
pair_rdd = transaction_rdd.map(
    lambda x: (x["Customer_ID"], x["Transaction_Amount"])
)

pair_rdd.take(10)

[('d5f6ec07-d69e-4f47-b9b4-7c58ff17c19e', 32415.45),
 ('7c14ad51-781a-4db9-b7bd-67439c175262', 43622.6),
 ('3a73a0e5-d4da-45aa-85f3-528413900a35', 63062.56),
 ('7902f4ef-9050-4a79-857d-9c2ea3181940', 14000.72),
 ('3a4bba70-d9a9-4c5f-8b92-1735fd8c19e9', 18335.16),
 ('6c870d65-76b0-431d-bdf3-9292998e8211', 9711.15),
 ('5323737c-bbd2-423f-9c9b-e0433c8f75dc', 94677.01),
 ('c0c3d474-f6c2-4c66-9b0e-f9ba75c6f310', 67704.28),
 ('e9a82764-1253-4a46-ad34-80e3416fc801', 72953.45),
 ('708224d5-192a-4d86-b411-8ec6d1bb274b', 5689.02)]

In [17]:
# Total Transaction Amount per Customer (reduceByKey)

customer_total = pair_rdd.reduceByKey(lambda x, y: x + y)

customer_total.take(10)

[('7c14ad51-781a-4db9-b7bd-67439c175262', 43622.6),
 ('3a73a0e5-d4da-45aa-85f3-528413900a35', 63062.56),
 ('7902f4ef-9050-4a79-857d-9c2ea3181940', 14000.72),
 ('3a4bba70-d9a9-4c5f-8b92-1735fd8c19e9', 18335.16),
 ('5323737c-bbd2-423f-9c9b-e0433c8f75dc', 94677.01),
 ('c0c3d474-f6c2-4c66-9b0e-f9ba75c6f310', 67704.28),
 ('e9a82764-1253-4a46-ad34-80e3416fc801', 72953.45),
 ('708224d5-192a-4d86-b411-8ec6d1bb274b', 5689.02),
 ('87b53799-9959-4534-9aad-f7dfc1821d4b', 45491.61),
 ('d4d0f62e-00b0-4c0e-8188-0b5a3f3e6c3e', 34680.87)]

In [18]:
#Sort by Customer ID

sorted_customers = customer_total.sortByKey()

sorted_customers.take(10)

[('00009fec-323d-4d5d-8919-25be9fb648bd', 23466.43),
 ('00058767-bab8-4ce0-8c84-b505935dc1ce', 23026.48),
 ('0005dd6e-5f6a-410d-8e2c-6b9f0afed22b', 18519.27),
 ('0005e813-1f35-4149-a52e-d9f6d0a7a1a1', 12944.62),
 ('0006419f-5470-475f-9649-1d00e3c560b7', 40638.95),
 ('0006ae15-bb04-4638-aeb2-3b76e5c8335c', 11476.0),
 ('00084fe6-4f15-472e-9c98-f1d9a16ee1c0', 73178.7),
 ('00096b3f-7a60-4922-8668-876cc47dfc6a', 12254.33),
 ('000acac1-fe09-4ecf-9184-58b0c6a5c38a', 58069.31),
 ('000d2ba5-f0cc-4aff-a798-0ec218adf1e8', 28385.96)]

In [19]:
from pyspark import StorageLevel

customer_total.persist(StorageLevel.MEMORY_AND_DISK)

print(customer_total.count())

61398


Q4) Spark DataFrame Operations

In [20]:
df.select(
    "Customer_ID",
    "Transaction_Amount",
    "Transaction_Type"
).show(10)

+--------------------+------------------+----------------+
|         Customer_ID|Transaction_Amount|Transaction_Type|
+--------------------+------------------+----------------+
|d5f6ec07-d69e-4f4...|          32415.45|        Transfer|
|7c14ad51-781a-4db...|           43622.6|    Bill Payment|
|3a73a0e5-d4da-45a...|          63062.56|    Bill Payment|
|7902f4ef-9050-4a7...|          14000.72|           Debit|
|3a4bba70-d9a9-4c5...|          18335.16|        Transfer|
|6c870d65-76b0-431...|           9711.15|        Transfer|
|5323737c-bbd2-423...|          94677.01|        Transfer|
|c0c3d474-f6c2-4c6...|          67704.28|           Debit|
|e9a82764-1253-4a4...|          72953.45|      Withdrawal|
|708224d5-192a-4d8...|           5689.02|          Credit|
+--------------------+------------------+----------------+
only showing top 10 rows


In [21]:
#Filter High Value Transactions  - Filter


high_value = df.filter(
    df.Transaction_Amount > 50000
)

high_value.show(10,truncate=False)

+------------------------------------+---------------+------+---+----------------------------------------+----------+-----------------+------------+------------------------------------+----------------+-------------------+------------------+------------------------------------+----------------+-----------------+---------------+----------------------------+--------------------------------------------------+-----------+--------+--------------------+----------------+--------------------------+------------------------+
|Customer_ID                         |Customer_Name  |Gender|Age|State                                   |City      |Bank_Branch      |Account_Type|Transaction_ID                      |Transaction_Date|Transaction_Time   |Transaction_Amount|Merchant_ID                         |Transaction_Type|Merchant_Category|Account_Balance|Transaction_Device          |Transaction_Location                              |Device_Type|Is_Fraud|Transaction_Currency|Customer_Contact|Transaction

In [22]:
#Branch-wise Total Amount  -  Grouping

from pyspark.sql.functions import sum

branch_amount = df.groupBy(
    "Bank_Branch"
).agg(
    sum("Transaction_Amount").alias("Total_Amount")
)

branch_amount.show()

+------------------+--------------------+
|       Bank_Branch|        Total_Amount|
+------------------+--------------------+
|       Kota Branch|1.6023036390000004E7|
|   Dehradun Branch|1.8693533289999995E7|
|    Lunglei Branch| 2.169940330000001E7|
|      Ajmer Branch|1.8398794040000003E7|
|    Jodhpur Branch|1.6354008799999999E7|
|      Jowai Branch| 2.377609391000001E7|
|       Mahe Branch| 2.167128004000001E7|
|   Diglipur Branch|2.9069315250000004E7|
|       Pune Branch|1.7165494119999997E7|
|     Bhopal Branch|1.6313926679999992E7|
|     Ujjain Branch|1.8549636459999993E7|
|Bhubaneswar Branch|1.6524616720000004E7|
|  Nongstoin Branch|2.4141323830000002E7|
|Muzaffarpur Branch|1.7410824630000003E7|
|    Dimapur Branch|       2.273031521E7|
|  Mangalore Branch|1.7065218569999993E7|
|   Durgapur Branch|1.9079259999999993E7|
|    Gwalior Branch|       1.680432183E7|
|      Patna Branch|       1.740135004E7|
|    Gangtok Branch|       1.977650105E7|
+------------------+--------------

In [23]:
customer_total = df.groupBy(
    "Customer_ID"
).agg(
    sum("Transaction_Amount").alias("Total_Amount")
)

joined_df = df.join(
    customer_total,
    on="Customer_ID",
    how="inner"
)

joined_df.select(
    "Customer_ID",
    "Transaction_Amount",
    "Total_Amount"
).show(10)

+--------------------+------------------+------------+
|         Customer_ID|Transaction_Amount|Total_Amount|
+--------------------+------------------+------------+
|32a50238-e369-442...|          80082.34|    80082.34|
|99db16b0-584b-4bf...|          67963.39|    67963.39|
|1ca810ff-c1c1-42b...|          31435.89|    31435.89|
|bfdc780f-845d-46e...|          92998.89|    92998.89|
|7c9049ce-656f-49c...|          40429.36|    40429.36|
|317713a1-1631-4cc...|            654.81|      654.81|
|8867e512-efee-46a...|           79749.5|     79749.5|
|1d55220e-2ea8-41c...|           96581.8|     96581.8|
|b1429f2b-f7d9-4f6...|          54003.67|    54003.67|
|a96438e7-dddf-4c3...|          16794.08|    16794.08|
+--------------------+------------------+------------+
only showing top 10 rows


In [24]:
from pyspark.sql.functions import sum, avg, max, min

df.groupBy(
    "Transaction_Type"
).agg(
    sum("Transaction_Amount").alias("Total"),
    avg("Transaction_Amount").alias("Average"),
    max("Transaction_Amount").alias("Maximum"),
    min("Transaction_Amount").alias("Minimum")
).show()

+----------------+-------------------+------------------+--------+-------+
|Transaction_Type|              Total|           Average| Maximum|Minimum|
+----------------+-------------------+------------------+--------+-------+
|    Bill Payment|6.051188401400006E8| 49349.11434839346|98994.49|  19.48|
|        Transfer|6.125236370399978E8| 49657.36822375337|98999.98|  10.52|
|      Withdrawal|6.045224495599983E8| 49599.80715129621|98999.02|  12.61|
|          Credit| 6.10298809970001E8| 49702.64760729709|98995.37|  20.66|
|           Debit|6.095972979399995E8|49424.136366142324|98970.43|  10.41|
+----------------+-------------------+------------------+--------+-------+



Q5) Exploratory Data Analysis and Spark SQL

In [25]:
df.createOrReplaceTempView("transactions")

In [26]:
# high_value = df.filter(df.Transaction_Amount > 50000)

# high_value.show(10)

In [27]:
spark.sql("""
SELECT *
FROM transactions
WHERE Transaction_Amount > 50000
""").show(10)

+--------------------+---------------+------+---+--------------------+----------+-----------------+------------+--------------------+----------------+-------------------+------------------+--------------------+----------------+-----------------+---------------+--------------------+--------------------+-----------+--------+--------------------+----------------+-----------------------+--------------------+
|         Customer_ID|  Customer_Name|Gender|Age|               State|      City|      Bank_Branch|Account_Type|      Transaction_ID|Transaction_Date|   Transaction_Time|Transaction_Amount|         Merchant_ID|Transaction_Type|Merchant_Category|Account_Balance|  Transaction_Device|Transaction_Location|Device_Type|Is_Fraud|Transaction_Currency|Customer_Contact|Transaction_Description|      Customer_Email|
+--------------------+---------------+------+---+--------------------+----------+-----------------+------------+--------------------+----------------+-------------------+--------------

In [28]:
# import pyspark.sql.functions as F

# customer_pattern = df.groupBy("Customer_ID").agg(
#     F.count("*").alias("Total_Transactions"),
#     F.sum("Transaction_Amount").alias("Total_Amount"),
#     F.avg("Transaction_Amount").alias("Average_Amount")
# )

# customer_pattern.show(10)

In [29]:
spark.sql("""
SELECT
Customer_ID,
COUNT(*) AS Total_Transactions,
SUM(Transaction_Amount) AS Total_Amount,
AVG(Transaction_Amount) AS Average_Amount
FROM transactions
GROUP BY Customer_ID
""").show(10)

+--------------------+------------------+------------+--------------+
|         Customer_ID|Total_Transactions|Total_Amount|Average_Amount|
+--------------------+------------------+------------+--------------+
|32a50238-e369-442...|                 1|    80082.34|      80082.34|
|99db16b0-584b-4bf...|                 1|    67963.39|      67963.39|
|1ca810ff-c1c1-42b...|                 1|    31435.89|      31435.89|
|bfdc780f-845d-46e...|                 1|    92998.89|      92998.89|
|7c9049ce-656f-49c...|                 1|    40429.36|      40429.36|
|317713a1-1631-4cc...|                 1|      654.81|        654.81|
|8867e512-efee-46a...|                 1|     79749.5|       79749.5|
|1d55220e-2ea8-41c...|                 1|     96581.8|       96581.8|
|b1429f2b-f7d9-4f6...|                 1|    54003.67|      54003.67|
|a96438e7-dddf-4c3...|                 1|    16794.08|      16794.08|
+--------------------+------------------+------------+--------------+
only showing top 10 

In [30]:
# branch_volume = df.groupBy("Bank_Branch").agg(
#     F.count("*").alias("Transaction_Count"),
#     F.sum("Transaction_Amount").alias("Total_Amount")
# )

# branch_volume.show()

In [31]:
spark.sql("""
SELECT
Bank_Branch,
COUNT(*) AS Transaction_Count,
SUM(Transaction_Amount) AS Total_Amount
FROM transactions
GROUP BY Bank_Branch
""").show()

+------------------+-----------------+--------------------+
|       Bank_Branch|Transaction_Count|        Total_Amount|
+------------------+-----------------+--------------------+
|       Kota Branch|              338|1.6023036390000004E7|
|   Dehradun Branch|              367|1.8693533289999995E7|
|    Lunglei Branch|              457| 2.169940330000001E7|
|      Ajmer Branch|              377|1.8398794040000003E7|
|    Jodhpur Branch|              350|1.6354008799999999E7|
|      Jowai Branch|              464| 2.377609391000001E7|
|       Mahe Branch|              445| 2.167128004000001E7|
|   Diglipur Branch|              588|2.9069315250000004E7|
|       Pune Branch|              356|1.7165494119999997E7|
|     Bhopal Branch|              343|1.6313926679999992E7|
|     Ujjain Branch|              363|1.8549636459999993E7|
|Bhubaneswar Branch|              334|1.6524616720000004E7|
|  Nongstoin Branch|              487|2.4141323830000002E7|
|Muzaffarpur Branch|              365|1.

In [32]:
# suspicious = df.filter(
#     (df.Transaction_Amount > 80000) &
#     (df.Is_Fraud == 1)
# )

# suspicious.show()

In [33]:
spark.sql("""
SELECT *
FROM transactions
WHERE
Transaction_Amount > 80000
AND Is_Fraud = 1
""").show()

+--------------------+-----------------+------+---+--------------------+-------------+--------------------+------------+--------------------+----------------+-------------------+------------------+--------------------+----------------+-----------------+---------------+--------------------+--------------------+-----------+--------+--------------------+----------------+-----------------------+--------------------+
|         Customer_ID|    Customer_Name|Gender|Age|               State|         City|         Bank_Branch|Account_Type|      Transaction_ID|Transaction_Date|   Transaction_Time|Transaction_Amount|         Merchant_ID|Transaction_Type|Merchant_Category|Account_Balance|  Transaction_Device|Transaction_Location|Device_Type|Is_Fraud|Transaction_Currency|Customer_Contact|Transaction_Description|      Customer_Email|
+--------------------+-----------------+------+---+--------------------+-------------+--------------------+------------+--------------------+----------------+----------

In [34]:
df.printSchema()

root
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- State: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Bank_Branch: string (nullable = true)
 |-- Account_Type: string (nullable = true)
 |-- Transaction_ID: string (nullable = true)
 |-- Transaction_Date: string (nullable = true)
 |-- Transaction_Time: timestamp (nullable = true)
 |-- Transaction_Amount: double (nullable = true)
 |-- Merchant_ID: string (nullable = true)
 |-- Transaction_Type: string (nullable = true)
 |-- Merchant_Category: string (nullable = true)
 |-- Account_Balance: double (nullable = true)
 |-- Transaction_Device: string (nullable = true)
 |-- Transaction_Location: string (nullable = true)
 |-- Device_Type: string (nullable = true)
 |-- Is_Fraud: integer (nullable = true)
 |-- Transaction_Currency: string (nullable = true)
 |-- Customer_Contact: string (nullable = true)
 |-- 

In [35]:
from pyspark.sql.functions import to_date

df = df.withColumn(
    "Transaction_Date",
    to_date("Transaction_Date", "dd-MM-yyyy")
)

Q6. ETL Pipeline Development

In [36]:
# ==========================
# ETL - EXTRACT
# ==========================

# Read the banking transaction dataset

extract_df = spark.read.csv(
    "Bank_Transaction_Fraud_Detection.csv",
    header=True,
    inferSchema=True
)

print("Total Records Extracted:", extract_df.count())

extract_df.show(5)

extract_df.printSchema()

Total Records Extracted: 135087
+--------------------+-------------------+------+---+-----------+------------------+--------------------+------------+--------------------+----------------+-------------------+------------------+--------------------+----------------+-----------------+---------------+------------------+--------------------+-----------+--------+--------------------+----------------+-----------------------+--------------------+
|         Customer_ID|      Customer_Name|Gender|Age|      State|              City|         Bank_Branch|Account_Type|      Transaction_ID|Transaction_Date|   Transaction_Time|Transaction_Amount|         Merchant_ID|Transaction_Type|Merchant_Category|Account_Balance|Transaction_Device|Transaction_Location|Device_Type|Is_Fraud|Transaction_Currency|Customer_Contact|Transaction_Description|      Customer_Email|
+--------------------+-------------------+------+---+-----------+------------------+--------------------+------------+--------------------+-----

In [37]:
# ==========================
# ETL - TRANSFORM
# ==========================

import pyspark.sql.functions as F
from pyspark.sql.functions import to_date, when

transform_df = extract_df

# Remove duplicate records
transform_df = transform_df.dropDuplicates()

# Remove rows with null values in important columns
transform_df = transform_df.dropna(
    subset=[
        "Customer_ID",
        "Transaction_ID",
        "Transaction_Amount",
        "Transaction_Date"
    ]
)

# Convert Transaction_Date to Date format
transform_df = transform_df.withColumn(
    "Transaction_Date",
    to_date("Transaction_Date", "dd-MM-yyyy")
)

# Add Transaction Category
transform_df = transform_df.withColumn(
    "Transaction_Category",
    when(
        F.col("Transaction_Amount") > 50000,
        "High Value"
    ).otherwise("Normal")
)

# Add Fraud Label
transform_df = transform_df.withColumn(
    "Fraud_Label",
    when(
        F.col("Is_Fraud") == 1,
        "Fraud"
    ).otherwise("Legitimate")
)

# Extract Month
transform_df = transform_df.withColumn(
    "Transaction_Month",
    F.month("Transaction_Date")
)

print("Records After Transformation:", transform_df.count())

transform_df.show(10)

Records After Transformation: 135087
+--------------------+----------------+------+---+--------------------+----------+-----------------+------------+--------------------+----------------+-------------------+------------------+--------------------+----------------+-----------------+---------------+--------------------+--------------------+-----------+--------+--------------------+----------------+-----------------------+--------------------+--------------------+-----------+-----------------+
|         Customer_ID|   Customer_Name|Gender|Age|               State|      City|      Bank_Branch|Account_Type|      Transaction_ID|Transaction_Date|   Transaction_Time|Transaction_Amount|         Merchant_ID|Transaction_Type|Merchant_Category|Account_Balance|  Transaction_Device|Transaction_Location|Device_Type|Is_Fraud|Transaction_Currency|Customer_Contact|Transaction_Description|      Customer_Email|Transaction_Category|Fraud_Label|Transaction_Month|
+--------------------+----------------+----

In [38]:
import os

# Windows Spark/Hadoop fix: point Spark to a local Hadoop home containing winutils.exe
# If you don't have winutils.exe, download it and place it under C:\hadoop\bin
os.environ['HADOOP_HOME'] = r'C:\hadoop'
os.environ['hadoop.home.dir'] = r'C:\hadoop'
os.environ['PATH'] = os.environ.get('PATH', '') + os.pathsep + os.path.join(os.environ['HADOOP_HOME'], 'bin')

print('HADOOP_HOME set to:', os.environ['HADOOP_HOME'])

HADOOP_HOME set to: C:\hadoop


In [39]:
import os

print(os.path.exists(r"C:\hadoop\bin\winutils.exe"))

False


In [40]:
# # ==========================
# # ETL - LOAD
# # ==========================

# # Save transformed data as CSV
# transform_df.write.mode("overwrite") \
#     .option("header", True) \
#     .csv("etl_csv")

# # Save transformed data as Parquet
# transform_df.write.mode("overwrite") \
#     .parquet("etl_parquet")

# print("ETL Pipeline Executed Successfully!")

# # Verify loaded data
# loaded_df = spark.read.parquet("etl_parquet")

# print("Loaded Records:", loaded_df.count())

# loaded_df.show(5)

In [42]:
output_directory = "/content/etl_parquet_output"
transform_df.write.mode("overwrite").parquet(output_directory)

Q7. Machine Learning/Deep Learning Implementation (2 Marks)
Implement a Machine Learning/Deep Learning model to predict fraudulent transactions or
customer churn and evaluate its performance.

In [43]:
# ==========================================
# Q7 - Feature Engineering
# ==========================================

from pyspark.ml.feature import StringIndexer, VectorAssembler

# Convert categorical columns into numeric indices
account_indexer = StringIndexer(
    inputCol="Account_Type",
    outputCol="Account_Type_Index",
    handleInvalid="keep"
)

transaction_indexer = StringIndexer(
    inputCol="Transaction_Type",
    outputCol="Transaction_Type_Index",
    handleInvalid="keep"
)

device_indexer = StringIndexer(
    inputCol="Device_Type",
    outputCol="Device_Type_Index",
    handleInvalid="keep"
)

branch_indexer = StringIndexer(
    inputCol="Bank_Branch",
    outputCol="Bank_Branch_Index",
    handleInvalid="keep"
)

# Fit and transform
df_ml = account_indexer.fit(df).transform(df)
df_ml = transaction_indexer.fit(df_ml).transform(df_ml)
df_ml = device_indexer.fit(df_ml).transform(df_ml)
df_ml = branch_indexer.fit(df_ml).transform(df_ml)

# Combine features into a single vector
assembler = VectorAssembler(
    inputCols=[
        "Age",
        "Transaction_Amount",
        "Account_Balance",
        "Account_Type_Index",
        "Transaction_Type_Index",
        "Device_Type_Index",
        "Bank_Branch_Index"
    ],
    outputCol="features"
)

df_ml = assembler.transform(df_ml)

df_ml.select("features", "Is_Fraud").show(5, truncate=False)

+------------------------------------------+--------+
|features                                  |Is_Fraud|
+------------------------------------------+--------+
|[60.0,32415.45,74557.27,2.0,0.0,3.0,70.0] |0       |
|[51.0,43622.6,74622.66,0.0,3.0,1.0,82.0]  |0       |
|[20.0,63062.56,66817.99,2.0,3.0,1.0,134.0]|0       |
|[57.0,14000.72,58177.08,0.0,1.0,2.0,74.0] |0       |
|[43.0,18335.16,16108.56,2.0,0.0,2.0,60.0] |0       |
+------------------------------------------+--------+
only showing top 5 rows


In [44]:
# ==========================================
# Train Logistic Regression Model
# ==========================================

from pyspark.ml.classification import LogisticRegression

# Split data into training and testing
train_data, test_data = df_ml.randomSplit([0.8, 0.2], seed=42)

# Logistic Regression Model
lr = LogisticRegression(
    featuresCol="features",
    labelCol="Is_Fraud"
)

# Train the model
model = lr.fit(train_data)

print("Model Training Completed")

Model Training Completed


In [45]:
# ==========================================
# Prediction and Evaluation
# ==========================================

from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Predict
predictions = model.transform(test_data)

predictions.select(
    "Is_Fraud",
    "prediction",
    "probability"
).show(10, truncate=False)

# Accuracy
accuracy = MulticlassClassificationEvaluator(
    labelCol="Is_Fraud",
    predictionCol="prediction",
    metricName="accuracy"
).evaluate(predictions)

# AUC
auc = BinaryClassificationEvaluator(
    labelCol="Is_Fraud",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
).evaluate(predictions)

print("Accuracy :", accuracy)
print("AUC Score:", auc)

+--------+----------+-----------------------------------------+
|Is_Fraud|prediction|probability                              |
+--------+----------+-----------------------------------------+
|0       |0.0       |[0.9482894031653328,0.051710596834667166]|
|0       |0.0       |[0.951608627421703,0.048391372578296954] |
|0       |0.0       |[0.9502912750718951,0.04970872492810485] |
|0       |0.0       |[0.9486469376967562,0.05135306230324377] |
|0       |0.0       |[0.9473842202800417,0.052615779719958344]|
|0       |0.0       |[0.9492350567839513,0.05076494321604874] |
|0       |0.0       |[0.9513857053720688,0.04861429462793121] |
|0       |0.0       |[0.9505946807304297,0.04940531926957026] |
|0       |0.0       |[0.9511109521974588,0.04888904780254122] |
|0       |0.0       |[0.9482399587454129,0.05176004125458711] |
+--------+----------+-----------------------------------------+
only showing top 10 rows
Accuracy : 0.9487706603075405
AUC Score: 0.5013252322495253
